# Snap Distance Append

## Centroid-to-Network-Node Distance for all_access

**Tess Vu**

This notebook appends two columns to the existing `tess_all_access.csv`:

- `walk_snap_dist_m` are meters from each SAL centroid to its nearest walk-network node.
- `drive_snap_dist_m` are meters from each SAL centroid to its nearest drive-network node.

No 2SFCA or Dijkstra computation is re-run.
The bottleneck is loading the four `.graphml` files from disk (~10–20 min total).
The snap computation itself (KDTree nearest-node + Euclidean distance) is instant.

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import osmnx as ox
import os
import warnings

warnings.filterwarnings("ignore")

## CONFIGURATION

In [2]:
# PATH TO THE EXISTING ALL_ACCESS CSV.
ALL_ACCESS_PATH = "data/tess_all_access.csv"

# SAL DEDUP SHAPEFILE FOR CENTROID GEOMETRY.
SAL_SHP_PATH = "data/sal_w_ward_dedup/sal_w_ward_dedup.shp"

# PRE-SAVED NETWORK GRAPH DIRECTORY.
NETWORK_DIR = "data/networks"

# OUTPUT PATH.
OUTPUT_PATH = "data/tess_all_access_w_snap.csv"

# PROJECTED CRS MATCHING THE 2SFCA NOTEBOOK.
PROJECTED_CRS = "EPSG:32735"

# GRAPH PATHS KEYED AS "Province_mode" TO MATCH all_access PR_NAME VALUES.
GRAPHML_PATHS = {
    "Gauteng_walk": os.path.join(NETWORK_DIR, "network_gauteng_walk.graphml"),
    "Gauteng_drive": os.path.join(NETWORK_DIR, "network_gauteng_drive.graphml"),
    "KwaZulu-Natal_walk": os.path.join(NETWORK_DIR, "network_kwazulu_natal_walk.graphml"),
    "KwaZulu-Natal_drive": os.path.join(NETWORK_DIR, "network_kwazulu_natal_drive.graphml"),
}

print(f"ALL_ACCESS INPUT: {ALL_ACCESS_PATH}")
print(f"SAL SHAPEFILE: {SAL_SHP_PATH}")
print(f"PROJECTED CRS: {PROJECTED_CRS}")

ALL_ACCESS INPUT: data/tess_all_access.csv
SAL SHAPEFILE: data/sal_w_ward_dedup/sal_w_ward_dedup.shp
PROJECTED CRS: EPSG:32735


## LOAD ALL_ACCESS AND COMPUTE SAL CENTROIDS

In [3]:
# Load the existing all_access CSV.
all_access = pd.read_csv(ALL_ACCESS_PATH, low_memory = False)
all_access["EA_CODE"] = pd.to_numeric(all_access["EA_CODE"], errors = "coerce").astype("Int64")

print(f"ALL_ACCESS LOADED: {len(all_access)} rows")
print(f"Columns with snap distance already: {[c for c in all_access.columns if 'snap' in c]}")

# Load SAL polygon shapefile and compute centroids in projected CRS.
# The centroid computation here matches exactly what the 2SFCA notebook does:
# sal.set_geometry(sal.geometry.centroid) after to_crs(PROJECTED_CRS).
sal_geo = gpd.read_file(SAL_SHP_PATH)
sal_geo["EA_CODE"] = pd.to_numeric(sal_geo["EA_CODE"], errors = "coerce").astype("Int64")

sal_proj = sal_geo.to_crs(PROJECTED_CRS)
sal_proj["centroid_x"] = sal_proj.geometry.centroid.x
sal_proj["centroid_y"] = sal_proj.geometry.centroid.y

print(f"SAL CENTROIDS COMPUTED: {len(sal_proj)} features")
print(f"CRS: {sal_proj.crs}")

ALL_ACCESS LOADED: 38380 rows
Columns with snap distance already: []
SAL CENTROIDS COMPUTED: 38380 features
CRS: EPSG:32735


In [4]:
# Build a lookup table from EA_CODE to centroid coordinates.
centroid_lookup = (
    sal_proj[["EA_CODE", "centroid_x", "centroid_y"]]
    .drop_duplicates(subset = "EA_CODE")
    .set_index("EA_CODE")
)

# Merge centroid coordinates onto all_access.
# Used only during this notebook as helper columns; dropped before saving.
all_access = all_access.merge(
    centroid_lookup,
    on = "EA_CODE",
    how = "left"
)

matched = all_access["centroid_x"].notna().sum()
print(f"CENTROID MERGE: {matched}/{len(all_access)} SALs matched")

CENTROID MERGE: 38380/38380 SALs matched


## COMPUTE SNAP DISTANCES PER PROVINCE AND NETWORK MODE

Each graph is loaded once, projected, and discarded.
`ox.nearest_nodes()` uses an internal KDTree — no routing is performed.
Snap distance is the straight-line Euclidean distance in meters from the SAL
centroid to the node it snapped to, computed in the projected CRS.

In [5]:
# Initialize output columns with NaN.
all_access["walk_snap_dist_m"] = np.nan
all_access["drive_snap_dist_m"] = np.nan

for graph_key, graphml_path in GRAPHML_PATHS.items():
    # Parse province name and mode from the key.
    province, mode = graph_key.rsplit("_", 1)
    col_name = f"{mode}_snap_dist_m"

    print(f"LOADING GRAPH: {graph_key}")

    # Load raw graph from disk, then project into metric CRS.
    G_raw = ox.load_graphml(graphml_path)
    G = ox.project_graph(G_raw, to_crs = PROJECTED_CRS)

    print(f"  Nodes: {G.number_of_nodes():,}  Edges: {G.number_of_edges():,}")

    # Extract node coordinates from the projected graph into a lookup DataFrame.
    # The projected graph stores node x and y in the metric CRS.
    node_coord_dict = {
        node_id: (data["x"], data["y"])
        for node_id, data in G.nodes(data = True)
    }
    node_x_arr = np.array([node_coord_dict[n][0] for n in node_coord_dict])
    node_y_arr = np.array([node_coord_dict[n][1] for n in node_coord_dict])
    node_id_arr = np.array(list(node_coord_dict.keys()))

    # Subset all_access to this province.
    province_mask = all_access["PR_NAME"] == province
    prov_df = all_access.loc[province_mask].copy()

    print(f"  SALs in {province}: {len(prov_df)}")

    # Drop any rows with missing centroid coordinates (uninhabited SALs with no geometry).
    valid_mask = prov_df["centroid_x"].notna() & prov_df["centroid_y"].notna()
    prov_valid = prov_df.loc[valid_mask]

    # Snap each SAL centroid to the nearest network node.
    # ox.nearest_nodes returns node IDs in the same order as the input coordinate arrays.
    snapped_node_ids = ox.nearest_nodes(
        G,
        prov_valid["centroid_x"].values,
        prov_valid["centroid_y"].values
    )

    # Look up the projected coordinates of each snapped node.
    snapped_x = np.array([node_coord_dict[n][0] for n in snapped_node_ids])
    snapped_y = np.array([node_coord_dict[n][1] for n in snapped_node_ids])

    # Compute Euclidean distance between SAL centroid and snapped node, in meters.
    snap_dist = np.sqrt(
        (prov_valid["centroid_x"].values - snapped_x) ** 2 +
        (prov_valid["centroid_y"].values - snapped_y) ** 2
    )

    # Write snap distances back to the correct rows of all_access.
    all_access.loc[prov_valid.index, col_name] = snap_dist

    print(f"  {col_name}: mean={snap_dist.mean():.0f}m, "
          f"median={np.median(snap_dist):.0f}m, "
          f"max={snap_dist.max():.0f}m, "
          f"flagged >500m: {(snap_dist > 500).sum()}")

    # Free graph memory before loading the next one.
    del G, G_raw
    print()

print("SNAP DISTANCE COMPUTATION COMPLETE")

LOADING GRAPH: Gauteng_walk
  Nodes: 443,832  Edges: 1,219,312
  SALs in Gauteng: 20850
  walk_snap_dist_m: mean=83m, median=57m, max=25702m, flagged >500m: 232

LOADING GRAPH: Gauteng_drive
  Nodes: 279,004  Edges: 730,792
  SALs in Gauteng: 20850
  drive_snap_dist_m: mean=101m, median=67m, max=26962m, flagged >500m: 398

LOADING GRAPH: KwaZulu-Natal_walk
  Nodes: 541,204  Edges: 1,356,318
  SALs in KwaZulu-Natal: 17530
  walk_snap_dist_m: mean=226m, median=83m, max=9951m, flagged >500m: 2149

LOADING GRAPH: KwaZulu-Natal_drive
  Nodes: 311,426  Edges: 749,194
  SALs in KwaZulu-Natal: 17530
  drive_snap_dist_m: mean=302m, median=102m, max=8435m, flagged >500m: 2884

SNAP DISTANCE COMPUTATION COMPLETE


## DIAGNOSTIC SUMMARY

In [6]:
# Summarize snap distance distributions across provinces and modes.
print("SNAP DISTANCE SUMMARY")
print()

for col in ["walk_snap_dist_m", "drive_snap_dist_m"]:
    mode = col.split("_")[0].capitalize()
    print(f"{mode} snap distances (meters):")
    for province in ["Gauteng", "KwaZulu-Natal"]:
        mask = all_access["PR_NAME"] == province
        vals = all_access.loc[mask, col].dropna()
        print(f"  {province}:")
        print(f"    mean={vals.mean():.0f}, median={np.median(vals):.0f}, "
              f"p95={np.percentile(vals, 95):.0f}, max={vals.max():.0f}")
        print(f"    SALs with snap > 100m: {(vals > 100).sum()} "
              f"({(vals > 100).mean() * 100:.1f}%)")
        print(f"    SALs with snap > 500m: {(vals > 500).sum()} "
              f"({(vals > 500).mean() * 100:.1f}%)")
        print(f"    SALs with snap > 1000m: {(vals > 1000).sum()} "
              f"({(vals > 1000).mean() * 100:.1f}%)")
    print()

SNAP DISTANCE SUMMARY

Walk snap distances (meters):
  Gauteng:
    mean=83, median=57, p95=223, max=25702
    SALs with snap > 100m: 4467 (21.4%)
    SALs with snap > 500m: 232 (1.1%)
    SALs with snap > 1000m: 41 (0.2%)
  KwaZulu-Natal:
    mean=226, median=83, p95=985, max=9951
    SALs with snap > 100m: 7559 (43.1%)
    SALs with snap > 500m: 2149 (12.3%)
    SALs with snap > 1000m: 852 (4.9%)

Drive snap distances (meters):
  Gauteng:
    mean=101, median=67, p95=290, max=26962
    SALs with snap > 100m: 5949 (28.5%)
    SALs with snap > 500m: 398 (1.9%)
    SALs with snap > 1000m: 91 (0.4%)
  KwaZulu-Natal:
    mean=302, median=102, p95=1318, max=8435
    SALs with snap > 100m: 8929 (50.9%)
    SALs with snap > 500m: 2884 (16.5%)
    SALs with snap > 1000m: 1381 (7.9%)



In [13]:
# Cross-tab of high snap distance by settlement type.
# SALs with snap > 500m are worth flagging in analysis.
SNAP_FLAG_THRESHOLD_M = 500

pop_data = pd.read_csv("data/pop_pred_final.csv")
pop_data["EA_CODE"] = pd.to_numeric(pop_data["EA_CODE"], errors = "coerce").astype("Int64")
pop_subset = pop_data[["EA_CODE", "EA_GTYPE", "EA_TYPE"]].copy()

# Drop any columns from all_access that overlap with pop_subset (excluding the merge key)
# to avoid pandas adding _x/_y suffixes.
overlap_cols = [c for c in ["EA_GTYPE", "EA_TYPE"] if c in all_access.columns]
diag = all_access.drop(columns = overlap_cols).merge(pop_subset, on = "EA_CODE", how = "left")

for col in ["walk_snap_dist_m", "drive_snap_dist_m"]:
    mode = col.split("_")[0]
    flag_col = f"{mode}_snap_flagged"
    diag[flag_col] = diag[col] > SNAP_FLAG_THRESHOLD_M

    print(f"{mode.upper()} SNAP FLAGS (>{SNAP_FLAG_THRESHOLD_M}m) BY SETTLEMENT TYPE")
    print(
        diag.groupby(["PR_NAME", "EA_GTYPE"])[flag_col]
        .agg(["sum", "count", "mean"])
        .rename(columns = {"sum": "n_flagged", "count": "n_total", "mean": "pct_flagged"})
        .assign(pct_flagged = lambda x: (x["pct_flagged"] * 100).round(1))
        .to_string()
    )
    print()

WALK SNAP FLAGS (>500m) BY SETTLEMENT TYPE
                           n_flagged  n_total  pct_flagged
PR_NAME       EA_GTYPE                                    
Gauteng       Farms               91      321         28.3
              Traditional          4      256          1.6
              Urban              137    20273          0.7
KwaZulu-Natal Farms              454      758         59.9
              Traditional       1617     7935         20.4
              Urban               78     8837          0.9

DRIVE SNAP FLAGS (>500m) BY SETTLEMENT TYPE
                           n_flagged  n_total  pct_flagged
PR_NAME       EA_GTYPE                                    
Gauteng       Farms              119      321         37.1
              Traditional          4      256          1.6
              Urban              275    20273          1.4
KwaZulu-Natal Farms              571      758         75.3
              Traditional       2158     7935         27.2
              Urban        

## SAVE

In [14]:
OUTPUT_PATH = "data/tess_all_access_w_snap.csv"

In [15]:
# Drop centroid helper columns before saving.
# These are already available in sal_pharmacy_distances_k3.csv.
all_access = all_access.drop(columns = ["centroid_x", "centroid_y"], errors = "ignore")

# Confirm final column additions.
print(f"NEW COLUMNS ADDED: walk_snap_dist_m, drive_snap_dist_m")
print(f"walk_snap_dist_m non-null: {all_access['walk_snap_dist_m'].notna().sum()}")
print(f"drive_snap_dist_m non-null: {all_access['drive_snap_dist_m'].notna().sum()}")
print(f"Total rows: {len(all_access)}")

# Overwrite all_access.
all_access.to_csv(OUTPUT_PATH, index = False)
print(f"SAVED: {OUTPUT_PATH}")

NEW COLUMNS ADDED: walk_snap_dist_m, drive_snap_dist_m
walk_snap_dist_m non-null: 38380
drive_snap_dist_m non-null: 38380
Total rows: 38380
SAVED: data/tess_all_access_w_snap.csv


## NOTES

- Snap distance is the **Euclidean** distance from the SAL centroid to the nearest
  network node in the projected CRS (EPSG:32735, units = meters). It is not a
  routed or network distance.

- Walk and drive snap distances differ because the walk and drive graphs have
  different node sets (different streets qualify for each mode).

- A snap distance of zero means the centroid projected exactly onto a network
  node, which is rare. Small values (< 50m) are normal in dense urban areas.

- Large snap distances (> 500m) indicate either a genuine data gap in OSM for
  that area, or a large SAL whose geometric centroid falls far from any mapped
  road. These are the same SALs where the 2SFCA Ai scores and k-nearest
  distances carry the most measurement uncertainty.

- These columns do not replace the `walk_circuity_k1` and `drive_circuity_k1`
  columns in `sal_pharmacy_distances_k3.csv`. Circuity captures how tortuous
  the actual routed path is; snap distance captures how far the path even started
  from the true centroid position.